# Great Expectations

## Why Do We Need Great Expectations?

Imagine you're a data engineer at a company. Every day, a CSV file with HDB resale flat transactions is delivered to your system. Your dashboards and ML models depend on this data being correct.

**What could go wrong?**
- The `resale_price` column suddenly contains negative values (data entry error)
- The `month` column is missing entirely (schema change upstream)
- The `flat_type` column has a new unexpected category like `"6 ROOM"` (new data)
- Thousands of rows have `null` values where there should be data

Without automated checks, these problems silently corrupt your analysis. You might only discover the issue days later — after reports have already been sent to stakeholders.

**Great Expectations solves this by letting you:**
1. **Define** what "good" data looks like (your Expectations)
2. **Validate** incoming data against those definitions automatically
3. **Alert** you (or stop a pipeline) when data doesn't meet your standards

Think of it like unit tests for your data. Just as `pytest` tests that your Python code behaves correctly, Great Expectations tests that your data behaves correctly.

---

## The Great Expectations Workflow

Here is the overall workflow we will follow in this notebook:

```
┌─────────────────────────────────────────────────────────────────┐
│                    GX Workflow Overview                         │
│                                                                 │
│  1. Data Context      ──► The project "home base" for all GX   │
│         │                                                       │
│  2. Data Source       ──► WHERE is your data? (filesystem, DB) │
│         │                                                       │
│  3. Data Asset        ──► WHAT collection of records? (a CSV)  │
│         │                                                       │
│  4. Batch             ──► Which SUBSET of records to validate? │
│         │                                                       │
│  5. Expectations      ──► Rules that define "good" data        │
│         │                                                       │
│  6. Expectation Suite ──► Bundle all rules together            │
│         │                                                       │
│  7. Validation        ──► Run the rules against the batch      │
│         │                                                       │
│  8. Checkpoint        ──► Automate validation + trigger actions│
└─────────────────────────────────────────────────────────────────┘
```

We will build each layer step by step below.

Great Expectations (GX) is a framework for describing data using expressive tests and then validating that the data meets test criteria. GX Core is a Python library that provides a programmatic interface to building and running data validation workflows using GX.

For more information, see here: https://docs.greatexpectations.io/docs/core/introduction/gx_overview

# Data Context 

Document: https://docs.greatexpectations.io/docs/core/set_up_a_gx_environment/create_a_data_context

> A Data Context defines the storage location for metadata, such as your configurations for Data Sources, Expectation Suites, Checkpoints, and Data Docs. It also contains your Validation Results and the metrics associated with them, and it provides access to those objects in Python, along with other helper functions for the GX Python API.

> All scripts that utilize GX Core should start with the creation of a Data Context.



### Step 1 — Create a Data Context

**What is it?**  
The Data Context is GX's central configuration object. Think of it like the "project folder" for all your data validation work. It keeps track of:
- Where your data lives (Data Sources)
- What your validation rules are (Expectation Suites)
- The history of past validation runs (Validation Results)

**`gx.get_context()`** creates (or loads) an **in-memory** context. This is the simplest option — nothing is saved to disk. For production use, you would use a persistent context that saves config to a file.

> **Key concept:** Every GX script starts with a Data Context. It is the entry point for all GX operations.

In [ ]:
import pandas as pd
import great_expectations as gx
from great_expectations import expectations as gxe

context = gx.get_context()

The `context` object is now your GX workspace. All subsequent steps will build on top of it.

---

## Step 2 — Connect to Your Data: Data Source

**What is it?**  
A Data Source tells GX *where* your data lives and *how* to connect to it. It does not load any data yet — it just registers the connection.

GX supports many types of data sources:
| Data Source Type | Example |
|---|---|
| Pandas Filesystem | CSV, Parquet, JSON files on your computer |
| SQL Database | PostgreSQL, MySQL, Snowflake |
| Spark | Large-scale distributed data |
| Cloud Storage | AWS S3, Azure Blob, Google Cloud |

In this lesson we use `add_pandas_filesystem`, which tells GX: *"Use pandas to read files from a local folder."*

**Parameters:**
- `name` — A friendly label you give this data source (you choose this)
- `base_directory` — The folder path where your CSV files are stored

Document: https://docs.greatexpectations.io/docs/core/connect_to_data/filesystem_data/

In [ ]:
source_folder = "../data/"
data_source_name = "resale_flat"

In [ ]:
data_source = context.data_sources.add_or_update_pandas_filesystem(
    name=data_source_name, 
    base_directory=source_folder
)

The `data_source` object is now registered in the context. GX knows to look in `../data/` for files, but hasn't read anything yet.

> **Note on re-running cells:** We use `add_or_update_pandas_filesystem` instead of `add_pandas_filesystem`. If you call `add_pandas_filesystem` a second time in the same kernel session (without restarting), GX raises a `DataContextError` because the name already exists. The `add_or_update` variant handles this gracefully — it creates the data source if it doesn't exist, or updates it if it does. You will see similar `try/except` patterns for assets and batch definitions below for the same reason.

---

## Step 3 — Identify Your Data: Data Asset

**What is it?**  
A Data Asset represents a specific *collection of records* within a Data Source. While the Data Source says "here is the folder", the Data Asset says "here is the specific type of file(s) I care about within that folder".

**Two types of Data Assets:**
- **File Data Asset** — Points to a single, specific file (e.g. one CSV)
- **Directory Data Asset** — Can match multiple files using a pattern (e.g. all CSVs in a folder)

Here we use `add_csv_asset`, which registers a CSV-type asset. At this point, GX still hasn't loaded any data — it just knows to expect CSV files.

**Think of it this way:**
- Data Source = the library building
- Data Asset = a specific bookshelf in that library
- Batch (next step) = a specific book you pull off that shelf

Documentation: https://docs.greatexpectations.io/docs/core/connect_to_data/filesystem_data/?data_asset=file#create-a-data-asset

In [ ]:
# we can retrieve as well
data_source = context.data_sources.get(data_source_name)

In [ ]:
asset_name = "resale_csv_files"

In [ ]:
try:
    file_csv_asset = data_source.add_csv_asset(name=asset_name)
except ValueError:
    file_csv_asset = data_source.get_asset(asset_name)

---

## Step 4 — Select Your Records: Batch Definition & Batch

**What is a Batch Definition?**  
A Batch Definition specifies *which records* from the Data Asset to load for validation. It is a reusable recipe: "give me the data from this specific file path".

**What is a Batch?**  
A Batch is the actual loaded data — the result of executing the Batch Definition. This is the object you run validations against.

**Why this separation?**  
In production, you might define "give me today's file" once, and then call `get_batch()` each day to get the current data. The definition stays the same; the data changes.

```
Batch Definition  ──► describes HOW to get data (the recipe)
      │
      ▼
   get_batch()
      │
      ▼
    Batch        ──► the actual data loaded into memory (the meal)
```

Below we use `add_batch_definition_path` to point at a specific CSV file by name:

In [ ]:
# we can retrive 
file_data_asset = context.data_sources.get(data_source_name).get_asset(asset_name)

In [ ]:
# batch_definition_name is the label GX uses to identify this definition internally.
# batch_definition_path is the actual filename GX uses to find the file on disk (relative to base_directory).
# They have the same value here by convention — naming the definition after the file it points to.
# You could use any friendly label for the name, e.g. "historical_resale_data", and it would still work.
batch_definition_name = "resale_flat_201701_202306.csv"
batch_definition_path = "resale_flat_201701_202306.csv"

try:
    batch_definition = file_data_asset.add_batch_definition_path(
        name=batch_definition_name, path=batch_definition_path
    )
except ValueError:
    batch_definition = file_data_asset.get_batch_definition(batch_definition_name)

In [ ]:
batch = batch_definition.get_batch()

`batch.head(4)` works just like a pandas DataFrame — because under the hood, GX uses pandas to load the data. You can now see the actual records you'll be validating.

In [ ]:
print(batch.head(4))

---

## Step 5 — Define Your Rules: Expectations

**What is an Expectation?**  
An Expectation is a *declarative statement* about what your data should look like. Instead of writing custom validation logic, you use GX's built-in Expectation classes.

**Examples of built-in Expectations:**

| Expectation | What it checks |
|---|---|
| `ExpectColumnToExist` | A column with this name exists |
| `ExpectColumnValuesToNotBeNull` | No null values in a column |
| `ExpectColumnValuesToBeBetween` | All values fall within a range |
| `ExpectColumnMaxToBeBetween` | The maximum value is within a range |
| `ExpectColumnValuesToBeInSet` | Values only come from an allowed list |
| `ExpectTableRowCountToBeBetween` | Row count is within expected bounds |

Browse the full gallery at: https://greatexpectations.io/expectations/

**Our first expectation:**  
We expect the maximum `lease_commence_date` to be between 1 and 2020. This catches data where a future year like 2099 was accidentally entered.

Document: https://docs.greatexpectations.io/docs/core/define_expectations/organize_expectation_suites

In [ ]:
preset_expectation = gx.expectations.ExpectColumnMaxToBeBetween(
    column="lease_commence_date", min_value=1, max_value=2020
)

In [ ]:
validation_results = batch.validate(preset_expectation)

In [ ]:
print(validation_results)

**Understanding the validation result:**  
The result object tells you:
- `"success": true` — the data met the expectation ✅
- `"observed_value": 2019` — the actual maximum value found was 2019 (within our range of 1–2020)
- `"expectation_config"` — a record of exactly what rule was checked

If the data had a `lease_commence_date` of 2025, the result would show `"success": false`, alerting you to the problem.

> **Quick validation vs. Suites:** The `batch.validate(expectation)` above is a quick one-off check. In real workflows, you group all your expectations into an **Expectation Suite** and validate them together — which we do next.

---

## Step 6 — Group Your Rules: Expectation Suite

**What is an Expectation Suite?**  
An Expectation Suite is a named collection of Expectations that all apply to the same dataset. Instead of running each Expectation individually, you bundle them into a Suite and run them all at once.

**Why use a Suite?**
- Run 20 checks with a single call instead of 20 separate calls
- Keep all the rules for a dataset in one place
- Re-use the same Suite across different validation runs
- Get a summary report: "17/20 expectations passed"

The workflow is:
1. Create an empty Suite with a name
2. Add it to the context (so it is registered/saved)
3. Add your Expectations to it one by one

> **Analogy:** If an Expectation is a single test case, an Expectation Suite is the full test file.

In [ ]:
suite_name = "sctp_expectation_suite"
suite = gx.ExpectationSuite(name=suite_name)

In [ ]:
suite = context.suites.add_or_update(suite)

In [ ]:
suite.add_expectation(preset_expectation)

# Rule 2: resale_price should never be null
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="resale_price")
)

# Rule 3: flat_type should only contain known categories
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="flat_type",
        value_set=["1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE", "MULTI-GENERATION"],
    )
)

Our suite now contains **3 expectations** — all run together in a single validation call:

| # | Expectation | What it checks |
|---|---|---|
| 1 | `ExpectColumnMaxToBeBetween` | `lease_commence_date` max is between 1 and 2020 |
| 2 | `ExpectColumnValuesToNotBeNull` | `resale_price` has no null values |
| 3 | `ExpectColumnValuesToBeInSet` | `flat_type` only contains the 7 known categories |

When you run the Validation Definition in Step 7, all 3 rules are evaluated at once and you get a single report showing how many passed. This is the core benefit of using a Suite over calling `batch.validate()` one expectation at a time.

---

## Step 7 — Run the Validation: Validation Definition

**What is a Validation Definition?**  
A Validation Definition links together:
- **What data** to validate (the Batch Definition)
- **What rules** to apply (the Expectation Suite)

It is the bridge that says: "Run *these rules* against *this data*."

Calling `.run()` on a Validation Definition executes all the expectations in the suite against the specified batch of data and returns a full results report.

**The results report includes:**
- `statistics` — a summary (how many expectations passed/failed)
- `results` — detailed per-expectation outcomes
- `success` — overall pass/fail (`true` only if ALL expectations passed)

See: https://docs.greatexpectations.io/docs/core/run_validations/create_a_validation_definition

In [ ]:
definition_name = "sctp_validation_definition"

try:
    validation_definition = context.validation_definitions.add(
        gx.ValidationDefinition(
            data=batch_definition, suite=suite, name=definition_name
        )
    )
except Exception:
    validation_definition = context.validation_definitions.get(definition_name)

In [ ]:
validation_results = validation_definition.run()

In [ ]:
print(validation_results)

**Reading the results:**
- `"evaluated_expectations": 3` — all 3 rules in the suite were checked
- `"successful_expectations": 3` — all 3 passed
- `"unsuccessful_expectations": 0` — no failures
- `"success_percent": 100.0` — 100% of rules passed
- `"success": true` — overall result is PASS ✅

The `results` array contains one entry per expectation, so you can drill into each rule individually:
- Rule 1 (`lease_commence_date` max): `observed_value: 2019` — within the 1–2020 range ✅
- Rule 2 (`resale_price` not null): `unexpected_count: 0` — zero nulls found ✅
- Rule 3 (`flat_type` in set): `unexpected_count: 0` — all values matched the allowed list ✅

In a pipeline, you would check `validation_results.success` — if `False`, stop the pipeline and alert the team before bad data reaches downstream systems.

---

## Seeing a Failure: What Does a Failed Validation Look Like?

So far all our validations have passed. Let's deliberately introduce a **failing expectation** so you know what to expect in a real pipeline when data quality issues are detected.

We'll add a new expectation that checks whether `flat_type` only contains values from a known set — but we'll intentionally leave out `"EXECUTIVE"` so the expectation fails.

> **Why this matters:** In production, you will catch real failures like this — a new data supplier adds a new category, or a column contains unexpected values after an upstream schema change.

In [ ]:
# Deliberately incomplete set — "EXECUTIVE" and "MULTI-GENERATION" are missing
failing_expectation = gx.expectations.ExpectColumnValuesToBeInSet(
    column="flat_type",
    value_set=["1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM"],
)

failing_result = batch.validate(failing_expectation)
print(failing_result)

**Reading a failed result:**
- `"success": false` — the expectation was NOT met ❌
- `"unexpected_list"` — shows the actual values that were not in the allowed set (e.g. `"EXECUTIVE"`, `"MULTI-GENERATION"`)
- `"unexpected_percent"` — what percentage of rows had an unexpected value

In a pipeline, you would check `if not validation_results.success: raise Exception("Data quality check failed")` to stop bad data from flowing downstream.

> **Fix it:** To make this expectation pass, add `"EXECUTIVE"` and `"MULTI-GENERATION"` to the `value_set` list above and re-run the cell.

---

## Step 8 — Automate & Act: Checkpoint

**What is a Checkpoint?**  
A Checkpoint is the final layer of the GX workflow. It takes one or more Validation Definitions and runs them together, then executes **Actions** based on the results.

**What are Actions?**  
Actions are automated responses to validation outcomes. Common examples:

| Action | What it does |
|---|---|
| `UpdateDataDocsAction` | Generates an HTML report you can browse in a browser |
| `SlackNotificationAction` | Posts a Slack message if validation fails |
| `EmailAction` | Sends an email alert on failure |
| `PagerDutyAlertAction` | Triggers an on-call alert |

**Why use a Checkpoint instead of just running Validation Definitions directly?**  
In production, you don't just want to *know* the result — you want the system to *respond* to it automatically. A Checkpoint lets you wire up those automated responses once, then trigger them every run.

```
              ┌──────────────────────────┐
              │       Checkpoint         │
              │                          │
              │  ValidationDefinition(s) │
              │         + Actions        │
              └──────────────┬───────────┘
                             │
                        .run()
                             │
               ┌─────────────┴─────────────┐
               │                           │
          PASS ✅                      FAIL ❌
               │                           │
     UpdateDataDocs              SlackNotification
                                 + StopPipeline
```

For more information, see here: https://docs.greatexpectations.io/docs/core/trigger_actions_based_on_results/create_a_checkpoint_with_actions/

> **Note:** `UpdateDataDocsAction` generates static HTML reports. This works best with a **file-system Data Context** (which persists config to disk). In this notebook we are using an **in-memory context**, so the action runs without error but the HTML files are written to a temporary location you won't easily find. In a real project, replace `gx.get_context()` with `gx.get_context(mode="file")` to get a persistent context that saves everything to a `gx/` folder.

In [ ]:
checkpoint_name = "sctp_checkpoint"

try:
    checkpoint = context.checkpoints.add(
        gx.Checkpoint(
            name=checkpoint_name,
            validation_definitions=[validation_definition],
            actions=[
                gx.checkpoint.UpdateDataDocsAction(name="update_data_docs"),
            ],
        )
    )
except Exception:
    checkpoint = context.checkpoints.get(checkpoint_name)

checkpoint_results = checkpoint.run()
print("Checkpoint passed:", checkpoint_results.success)
print("Statistics:", checkpoint_results.describe_dict())

---

## Bonus: Validating a Pandas DataFrame Directly

So far we connected GX to files on disk. But in many real workflows, you've already loaded data into a pandas DataFrame in memory — maybe after cleaning it or fetching it from an API.

GX supports this too with a **Pandas DataFrame Data Source**. Instead of pointing to a file path, you pass the DataFrame directly when fetching a batch.

**When to use this approach:**
- You loaded data from a database and want to validate it before processing
- You applied transformations and want to validate the result
- You're writing a notebook that combines analysis and validation

**Key difference from the filesystem approach:**

| Filesystem Data Source | Pandas DataFrame Data Source |
|---|---|
| Points to files on disk | Works with an in-memory DataFrame |
| File path is fixed at Batch Definition time | DataFrame is passed in at `get_batch()` time |
| Good for pipeline automation | Good for notebook-based validation |

In this example, let us try to use the other resale data.

In [ ]:
import pandas as pd
import great_expectations as gx
from great_expectations import expectations as gxe

In [ ]:
resale_data = pd.read_csv("../data/resale_flat_202307.csv")

In [ ]:
# https://docs.greatexpectations.io/docs/core/connect_to_data/dataframes/

# We create a FRESH context here intentionally.
# The earlier context already has a filesystem data source registered under "resale_flat".
# Adding a Pandas (in-memory) data source to the same context with a different name would work,
# but starting fresh keeps this bonus section self-contained and avoids name collisions
# if you re-run cells out of order.

# Step 1: Fresh context
context = gx.get_context()

# Step 2: Create a Pandas (in-memory) data source — not filesystem this time
data_source_name = "resale_dataframe"
data_source = context.data_sources.add_or_update_pandas(name=data_source_name)

# Step 3: Create a DataFrame asset — note "dataframe" asset type, not "csv"
data_asset_name = "resale_202307_asset"
try:
    data_asset = data_source.add_dataframe_asset(name=data_asset_name)
except ValueError:
    data_asset = data_source.get_asset(data_asset_name)

# Step 4: Create a batch definition — "whole dataframe" means no date partitioning
batch_definition_name = "resale_flat_202307_dataframe"
try:
    batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_definition_name)
except ValueError:
    batch_definition = data_asset.get_batch_definition(batch_definition_name)

In [ ]:
# Step 5: Pass the DataFrame in at get_batch() time — this is the key difference!
# The DataFrame is NOT baked into the definition; it is injected here.
batch_parameters = {"dataframe": resale_data}

new_batch = batch_definition.get_batch(batch_parameters=batch_parameters)

In [ ]:
print(new_batch.head(10))

In [ ]:
# ExpectColumnToExist verifies that the column is present in the dataset.
# column_index=0 additionally checks that "month" is the FIRST column (index 0).
# This is useful to detect schema drift — e.g. if someone reordered the columns.
new_expectation = gx.expectations.ExpectColumnToExist(column="month", column_index=0)

In [ ]:
validation_results = new_batch.validate(new_expectation)

In [ ]:
print(validation_results)

---

## Try It Yourself 🧪

Now it's your turn. Use the `new_batch` from the DataFrame section above to write and validate your own expectations.

**Challenge 1 — Check for nulls:**  
Write an expectation that verifies `resale_price` has no null values.
```python
# Your code here
expectation = gx.expectations.ExpectColumnValuesToNotBeNull(column="???")
result = new_batch.validate(expectation)
print(result)
```

**Challenge 2 — Check row count:**  
Write an expectation that verifies the dataset has at least 1,000 rows.
```python
# Your code here
expectation = gx.expectations.ExpectTableRowCountToBeGreaterThan(value=???)
result = new_batch.validate(expectation)
print(result)
```

**Challenge 3 — Check price range:**  
Write an expectation that `resale_price` values are between 100,000 and 2,000,000.
```python
# Your code here
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="resale_price", min_value=???, max_value=???
)
result = new_batch.validate(expectation)
print(result)
```

Browse more built-in expectations at: https://greatexpectations.io/expectations/

---

## Summary: The Full GX Workflow at a Glance

Here is everything we covered, condensed:

```python
import great_expectations as gx

# 1. Data Context — the project workspace
context = gx.get_context()

# 2. Data Source — WHERE is the data?
data_source = context.data_sources.add_or_update_pandas_filesystem(name="...", base_directory="...")

# 3. Data Asset — WHAT type of records?
try:
    asset = data_source.add_csv_asset(name="...")
except ValueError:
    asset = data_source.get_asset("...")

# 4. Batch Definition + Batch — WHICH records, loaded into memory
try:
    batch_definition = asset.add_batch_definition_path(name="...", path="file.csv")
except ValueError:
    batch_definition = asset.get_batch_definition("...")
batch = batch_definition.get_batch()

# 5. Expectations — WHAT rules must the data satisfy?
expectation = gx.expectations.ExpectColumnValuesToNotBeNull(column="resale_price")

# 6. Expectation Suite — bundle all rules together
suite = gx.ExpectationSuite(name="my_suite")
suite = context.suites.add_or_update(suite)
suite.add_expectation(expectation)

# 7. Validation Definition — link data + rules
try:
    validation_def = context.validation_definitions.add(
        gx.ValidationDefinition(data=batch_definition, suite=suite, name="...")
    )
except Exception:
    validation_def = context.validation_definitions.get("...")
results = validation_def.run()

# 8. Checkpoint — automate + trigger actions
try:
    checkpoint = context.checkpoints.add(
        gx.Checkpoint(name="...", validation_definitions=[validation_def], actions=[...])
    )
except Exception:
    checkpoint = context.checkpoints.get("...")
checkpoint.run()

# In a pipeline — stop on failure
if not results.success:
    raise Exception("Data quality check failed — pipeline halted.")
```

**Next steps to explore:**
- Browse all available Expectations: https://greatexpectations.io/expectations/
- Learn about Data Docs (auto-generated HTML reports): https://docs.greatexpectations.io/docs/core/trigger_actions_based_on_results/create_a_checkpoint_with_actions/
- Connect to a SQL database instead of a file: https://docs.greatexpectations.io/docs/core/connect_to_data/sql_data/
- Use a file-based context for persistent config: `gx.get_context(mode="file")`

Lastly, see expectations gallery for more examples!: https://greatexpectations.io/expectations/